# 02 — Asymmetric GPU Encoding SPECTER2 (FP16) & Konstruksi Tervalidasi `LitSearch-Neg` (Hari 2)

Notebook **siap jalan (*turnkey*)** untuk dieksekusi di **Google Colab (Runtime -> Change runtime type -> T4 GPU)**.

### Alur Eksekusi Otomatis (Tahap 2 & Tahap 3):
1. **Persiapan Lingkungan & Google Drive**: Memasang versi paket kompatibel (`transformers==4.57.6`, `adapters==1.3.0`, `datasets`), memeriksa GPU NVIDIA Tesla T4, dan menautkan Google Drive untuk *checkpoint* aman.
2. **Unduh & Validasi `princeton-nlp/LitSearch`**: Memuat `64.183` artikel CS/AI (`corpus_clean`) dan `597` kueri pencarian (`query`), identik 100% dengan `src/data_loader.py` lokal.
3. **Encoding Kueri Positif (`adhoc_query`)**: Mengenkode `597` kueri menggunakan adapter `allenai/specter2_adhoc_query` -> `q_pos_emb.npy` (`(597, 768)`, `float32`).
4. **Encoding Cepat `64.183` Dokumen (`proximity` + `FP16` + *Length-Sorted Dynamic Padding*)**: Mengenkode `Title + [SEP] + Abstract` menggunakan adapter `allenai/specter2` (`proximity`) dalam **~8–10 menit** lengkap *checkpoint* otomatis tiap `10.000` dokumen -> `doc_emb.npy` (`(64183, 768)`, `float32`).
5. **Konstruksi Tervalidasi 3 Lapis `LitSearch-Neg` & Split `seed=42`**: Membentuk pasangan `(Q_pos, Q_neg, violator_doc_ids)` bebas konflik (`0%` *gold overlap*) serta membagi `20%` Validation Split (`negative_queries_val.jsonl`) dan `80%` Held-Out Test Split (`negative_queries_test.jsonl`).
6. **Encoding Kueri Negatif & Pengemasan `day2_colab_outputs.zip`**: Mengenkode `q_neg_emb.npy` dan `q_naive_not_emb.npy` memakai `adhoc_query`, memverifikasi nol `NaN`/`Inf`, lalu membungkus seluruh hasil ke `day2_colab_outputs.zip` agar siap diunduh 1 klik ke laptop.

In [1]:
# === SEL 1: Instalasi Paket Terkunci & Mount Google Drive ===
import importlib
import subprocess
import sys
from pathlib import Path

print("[1/2] Memasang paket kompatibel (transformers==4.57.6, adapters==1.3.0, datasets)...")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers==4.57.6",
    "adapters==1.3.0",
    "datasets>=2.20.0",
])
importlib.invalidate_caches()

USE_GOOGLE_DRIVE = True   # Simpan otomatis ke Google Drive agar aman dari putus koneksi
FORCE_REENCODE = False    # Ubah ke True hanya jika ingin menghitung ulang dari awal

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        OUTPUT_ROOT = Path("/content/drive/MyDrive/metopen_specter2_litsearch_day2")
    except ImportError:
        OUTPUT_ROOT = Path.cwd() / "colab_outputs_day2"
else:
    OUTPUT_ROOT = Path("/content/metopen_specter2_litsearch_day2")

DATA_DIR = OUTPUT_ROOT / "data" / "litsearch"
EMBEDDINGS_DIR = OUTPUT_ROOT / "embeddings"
CHECKPOINTS_DIR = EMBEDDINGS_DIR / "checkpoints"
TABLES_DIR = OUTPUT_ROOT / "results" / "tables"

for folder in (DATA_DIR / "qrels", EMBEDDINGS_DIR, CHECKPOINTS_DIR, TABLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"[2/2] Direktori kerja aktif: {OUTPUT_ROOT}")

[1/2] Memasang paket kompatibel (transformers==4.57.6, adapters==1.3.0, datasets)...
Mounted at /content/drive
[2/2] Direktori kerja aktif: /content/drive/MyDrive/metopen_specter2_litsearch_day2


In [2]:
# === SEL 2: Verifikasi GPU Tesla T4 & Muat Dataset Resmi LitSearch ===
import csv
import json
import platform
import re
import time
from typing import Any

import adapters
import datasets
import numpy as np
import torch
import transformers
from adapters import AutoAdapterModel
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

assert torch.cuda.is_available(), (
    "GPU CUDA tidak terdeteksi! Di Google Colab klik: Runtime -> Change runtime type -> T4 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
env_info = {
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "transformers_version": transformers.__version__,
    "adapters_version": adapters.__version__,
    "datasets_version": datasets.__version__,
    "gpu_name": gpu_name,
    "gpu_vram_gb": vram_gb,
}
print("=== SPESIFIKASI LINGKUNGAN COLAB GPU ===")
for k, v in env_info.items():
    print(f"  {k:22s}: {v}")

EXPECTED_CORPUS_COUNT = 64183
EXPECTED_QUERY_COUNT = 597


def build_specter2_doc_text(doc: dict[str, Any], sep_token: str) -> str:
    title = (doc.get("title") or "").strip()
    abstract = (doc.get("text") or "").strip()
    return f"{title}{sep_token}{abstract}"


print("\nMemuat princeton-nlp/LitSearch dari Hugging Face...")
corpus_ds = load_dataset("princeton-nlp/LitSearch", "corpus_clean", split="full")
query_ds = load_dataset("princeton-nlp/LitSearch", "query", split="full")

corpus: dict[str, dict[str, Any]] = {}
doc_ids: list[str] = []
for row in corpus_ds:
    cid = str(row["corpusid"])
    doc_ids.append(cid)
    corpus[cid] = {
        "_id": cid,
        "title": (row.get("title") or ""),
        "text": (row.get("abstract") or ""),
    }

queries: dict[str, dict[str, Any]] = {}
qrels: dict[str, dict[str, int]] = {}
query_ids: list[str] = []
for idx, row in enumerate(query_ds):
    qid = f"q_{idx:03d}"
    gold_ids = [str(cid) for cid in row["corpusids"]]
    query_ids.append(qid)
    queries[qid] = {
        "_id": qid,
        "text": (row.get("query") or "").strip(),
        "specificity": int(row.get("specificity", 0)),
        "quality": int(row.get("quality", 0)),
        "query_set": str(row.get("query_set", "")),
        "gold_ids": gold_ids,
    }
    qrels[qid] = {cid: 1 for cid in dict.fromkeys(gold_ids)}

assert len(doc_ids) == EXPECTED_CORPUS_COUNT, f"Jumlah dokumen salah: {len(doc_ids)}"
assert len(query_ids) == EXPECTED_QUERY_COUNT, f"Jumlah kueri salah: {len(query_ids)}"
assert all(gid in corpus for gmap in qrels.values() for gid in gmap), "Ada gold ID hilang!"
print(f"[OK] LitSearch tervalidasi: {len(doc_ids)} dokumen, {len(query_ids)} kueri, 100% gold IDs cocok!")

=== SPESIFIKASI LINGKUNGAN COLAB GPU ===
  python_version        : 3.13.16
  torch_version         : 2.11.0+cu130
  cuda_version          : 13.0
  transformers_version  : 4.57.6
  adapters_version      : 1.3.0
  datasets_version      : 4.8.5
  gpu_name              : Tesla T4
  gpu_vram_gb           : 14.56

Memuat princeton-nlp/LitSearch dari Hugging Face...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

corpus_clean/full-00000-of-00006.parquet:   0%|          | 0.00/275M [00:00<?, ?B/s]

corpus_clean/full-00001-of-00006.parquet:   0%|          | 0.00/196M [00:00<?, ?B/s]

corpus_clean/full-00002-of-00006.parquet:   0%|          | 0.00/195M [00:00<?, ?B/s]

... (3 outputs left out)

Generating full split:   0%|          | 0/64183 [00:00<?, ? examples/s]

query/full-00000-of-00001.parquet:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

Generating full split:   0%|          | 0/597 [00:00<?, ? examples/s]

[OK] LitSearch tervalidasi: 64183 dokumen, 597 kueri, 100% gold IDs cocok!


In [3]:
# === SEL 3: Muat Model Asimetris SPECTER2 & Enkode 597 Kueri Positif (adhoc_query) ===
device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")

# Muat kedua adapter resmi, namun aktifkan secara eksklusif sesuai peran masing-masing
model.load_adapter("allenai/specter2_adhoc_query", source="hf", load_as="adhoc_query", set_active=False)
model.load_adapter("allenai/specter2", source="hf", load_as="proximity", set_active=False)
model.to(device)
model.eval()

# Aktifkan adapter adhoc_query untuk kueri positif Q_pos
model.set_active_adapters("adhoc_query")
active_query_adapter = str(model.active_adapters)
print("Adapter aktif untuk Q_pos:", active_query_adapter)
assert "adhoc_query" in active_query_adapter and "proximity" not in active_query_adapter, (
    f"GAGAL ASSERT ADAPTER KUERI! Terdeteksi: {active_query_adapter}"
)


def encode_texts_fp16(
    texts: list[str],
    batch_size: int = 64,
    desc: str = "Encoding",
) -> np.ndarray:
    outputs_list: list[np.ndarray] = []
    with torch.no_grad():
        for start in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[start : start + batch_size]
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):
                cls_emb = model(**enc).last_hidden_state[:, 0, :]
            outputs_list.append(cls_emb.float().cpu().numpy())
    return np.vstack(outputs_list).astype(np.float32)


# Warm-up singkat GPU
_ = encode_texts_fp16([queries["q_000"]["text"]], batch_size=1, desc="GPU Warm-up (Query)")
torch.cuda.synchronize()

t0_qpos = time.perf_counter()
q_pos_texts = [queries[qid]["text"] for qid in query_ids]
q_pos_emb = encode_texts_fp16(q_pos_texts, batch_size=64, desc="Encoding 597 Q_pos (adhoc_query)")
torch.cuda.synchronize()
q_pos_seconds = round(time.perf_counter() - t0_qpos, 3)

assert q_pos_emb.shape == (EXPECTED_QUERY_COUNT, 768)
assert np.isfinite(q_pos_emb).all()
assert float(np.linalg.norm(q_pos_emb, axis=1).min()) > 0.0

np.save(EMBEDDINGS_DIR / "q_pos_emb.npy", q_pos_emb)
with (EMBEDDINGS_DIR / "query_ids.json").open("w", encoding="utf-8") as f:
    json.dump(query_ids, f, indent=2)

print(f"[OK] q_pos_emb.npy tersimpan: shape={q_pos_emb.shape}, waktu={q_pos_seconds}s")

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

... (11 outputs left out)

Adapter aktif untuk Q_pos: Stack[adhoc_query]


GPU Warm-up (Query):   0%|          | 0/1 [00:00<?, ?it/s]

Encoding 597 Q_pos (adhoc_query):   0%|          | 0/10 [00:00<?, ?it/s]

[OK] q_pos_emb.npy tersimpan: shape=(597, 768), waktu=1.396s


In [4]:
# === SEL 4: Enkode Cepat 64.183 Dokumen Korpus (proximity + FP16 + Length-Sorted Checkpoints) ===
model.set_active_adapters("proximity")
active_doc_adapter = str(model.active_adapters)
print("Adapter aktif untuk Dokumen:", active_doc_adapter)
assert "proximity" in active_doc_adapter and "adhoc_query" not in active_doc_adapter, (
    f"GAGAL ASSERT ADAPTER DOKUMEN! Terdeteksi: {active_doc_adapter}"
)

doc_emb_path = EMBEDDINGS_DIR / "doc_emb.npy"
doc_ids_path = EMBEDDINGS_DIR / "doc_ids.json"

if doc_emb_path.exists() and doc_ids_path.exists() and not FORCE_REENCODE:
    print("[INFO] doc_emb.npy sudah tersedia di Drive, memuat ulang & memverifikasi...")
    doc_emb = np.load(doc_emb_path)
    saved_doc_ids = json.loads(doc_ids_path.read_text(encoding="utf-8"))
    assert saved_doc_ids == doc_ids, "Urutan doc_ids.json berbeda dari korpus!"
    assert doc_emb.shape == (EXPECTED_CORPUS_COUNT, 768) and np.isfinite(doc_emb).all()
    doc_encoding_seconds = -1.0
else:
    doc_texts = [
        build_specter2_doc_text(corpus[cid], sep_token=tokenizer.sep_token)
        for cid in doc_ids
    ]
    # Urutkan indeks berdasarkan panjang karakter agar dynamic padding per batch=128 sangat efisien
    char_lengths = np.array([len(t) for t in doc_texts], dtype=np.int32)
    sorted_indices = np.argsort(char_lengths, kind="mergesort")
    sorted_texts = [doc_texts[i] for i in sorted_indices]

    # Warm-up 1 batch untuk adapter proximity
    _ = encode_texts_fp16(sorted_texts[:32], batch_size=32, desc="GPU Warm-up (Doc)")
    torch.cuda.synchronize()

    BATCH_SIZE = 128
    CHECKPOINT_STEP = 10000
    sorted_chunks: list[np.ndarray] = []

    t0_doc = time.perf_counter()
    for ckpt_start in range(0, EXPECTED_CORPUS_COUNT, CHECKPOINT_STEP):
        ckpt_end = min(ckpt_start + CHECKPOINT_STEP, EXPECTED_CORPUS_COUNT)
        ckpt_file = CHECKPOINTS_DIR / f"doc_sorted_{ckpt_start:05d}_{ckpt_end:05d}.npy"

        if ckpt_file.exists() and not FORCE_REENCODE:
            print(f"[RESUME] Memuat checkpoint {ckpt_file.name}...")
            chunk_arr = np.load(ckpt_file)
        else:
            sub_texts = sorted_texts[ckpt_start:ckpt_end]
            chunk_arr = encode_texts_fp16(
                sub_texts,
                batch_size=BATCH_SIZE,
                desc=f"Docs {ckpt_start}..{ckpt_end} (proximity FP16)",
            )
            np.save(ckpt_file, chunk_arr)
            print(f"[CHECKPOINT] Tersimpan aman: {ckpt_file.name} {chunk_arr.shape}")

        sorted_chunks.append(chunk_arr)

    torch.cuda.synchronize()
    doc_encoding_seconds = round(time.perf_counter() - t0_doc, 3)

    sorted_emb = np.vstack(sorted_chunks).astype(np.float32)
    assert sorted_emb.shape == (EXPECTED_CORPUS_COUNT, 768)

    # Kembalikan baris matriks ke urutan asli doc_ids secara eksak
    doc_emb = np.empty_like(sorted_emb, dtype=np.float32)
    doc_emb[sorted_indices] = sorted_emb

    assert doc_emb.shape == (EXPECTED_CORPUS_COUNT, 768)
    assert np.isfinite(doc_emb).all(), "Ditemukan NaN/Inf pada doc_emb!"
    assert float(np.linalg.norm(doc_emb, axis=1).min()) > 0.0, "Ada vektor dokumen bernorma 0!"

    np.save(doc_emb_path, doc_emb)
    with doc_ids_path.open("w", encoding="utf-8") as f:
        json.dump(doc_ids, f, indent=2)

print(f"[OK] doc_emb.npy siap: shape={doc_emb.shape}, dtype={doc_emb.dtype}, waktu={doc_encoding_seconds}s")

Adapter aktif untuk Dokumen: Stack[proximity]


GPU Warm-up (Doc):   0%|          | 0/1 [00:00<?, ?it/s]

Docs 0..10000 (proximity FP16):   0%|          | 0/79 [00:00<?, ?it/s]

[CHECKPOINT] Tersimpan aman: doc_sorted_00000_10000.npy (10000, 768)


Docs 10000..20000 (proximity FP16):   0%|          | 0/79 [00:00<?, ?it/s]

[CHECKPOINT] Tersimpan aman: doc_sorted_10000_20000.npy (10000, 768)


... (6 outputs left out)

Docs 50000..60000 (proximity FP16):   0%|          | 0/79 [00:00<?, ?it/s]

[CHECKPOINT] Tersimpan aman: doc_sorted_50000_60000.npy (10000, 768)


Docs 60000..64183 (proximity FP16):   0%|          | 0/33 [00:00<?, ?it/s]

[CHECKPOINT] Tersimpan aman: doc_sorted_60000_64183.npy (4183, 768)
[OK] doc_emb.npy siap: shape=(64183, 768), dtype=float32, waktu=349.296s


In [5]:
# === SEL 5: Konstruksi Tervalidasi 3 Lapis LitSearch-Neg & Split Val (20%) / Test (80%) ===
SPLIT_SEED = 42
VAL_RATIO = 0.20
MIN_CONCEPT_DOCS = 15
MAX_CONCEPT_DOCS = 6000
TOP_K_DISTRACTORS = 5

# Kata kepala metodologis umum yang boleh berbagi kata umum selama kata pembeda utamanya tidak berkonflik
GENERIC_HEAD_TOKENS: frozenset[str] = frozenset({
    "learning", "model", "models", "modeling", "modelling",
    "network", "networks", "neural", "deep", "machine",
    "system", "systems", "task", "tasks", "method", "methods",
    "data", "dataset", "datasets", "training", "tuning",
    "generation", "detection", "analysis", "classification",
    "extraction", "parsing", "prediction", "recognition",
    "representation", "representations", "understanding",
    "processing", "natural", "language", "languages",
    "based", "level", "multi", "low", "open", "automatic",
    "inference", "optimization", "search", "evaluation",
    "estimation", "synthesis", "compression", "filtering",
    "transfer", "adaptation", "augmentation", "alignment",
    "retrieval", "matching", "scoring", "ranking", "embedding",
    "embeddings", "transformer", "transformers", "attention",
    "sequence", "text", "textual", "document", "documents",
    "sentence", "sentences", "word", "words", "token", "tokens",
})

# Taksonomi terkurasi konsep riset CS / ML / NLP / AI untuk ekstraksi Q_neg bebas noise boilerplate
CS_CONCEPT_LEXICON: tuple[str, ...] = (
    # NLP Tasks & Applications
    "machine translation",
    "neural machine translation",
    "simultaneous translation",
    "speech translation",
    "back-translation",
    "speech recognition",
    "automatic speech recognition",
    "speech synthesis",
    "text-to-speech",
    "speaker verification",
    "question answering",
    "visual question answering",
    "open-domain question answering",
    "reading comprehension",
    "named entity recognition",
    "entity linking",
    "entity typing",
    "relation extraction",
    "event extraction",
    "event detection",
    "information extraction",
    "open information extraction",
    "sentiment analysis",
    "aspect-based sentiment",
    "emotion recognition",
    "stance detection",
    "text summarization",
    "abstractive summarization",
    "extractive summarization",
    "multi-document summarization",
    "dialogue systems",
    "task-oriented dialogue",
    "conversational agents",
    "dialogue state tracking",
    "response generation",
    "semantic parsing",
    "dependency parsing",
    "constituency parsing",
    "part-of-speech tagging",
    "semantic role labeling",
    "coreference resolution",
    "word sense disambiguation",
    "natural language inference",
    "textual entailment",
    "paraphrase generation",
    "text simplification",
    "grammatical error correction",
    "style transfer",
    "code generation",
    "program synthesis",
    "code summarization",
    "vulnerability detection",
    "bug localization",
    "fact verification",
    "claim verification",
    "hate speech",
    "toxicity detection",
    "offensive language",
    "rumor detection",
    "fake news",
    "misinformation",
    "sarcasm detection",
    "metaphor detection",
    "humor detection",
    "topic modeling",
    "keyphrase generation",
    "keyphrase extraction",
    "optical character recognition",
    "sign language",
    "lip reading",
    "argument mining",
    "essay scoring",
    "readability assessment",
    # Architectures & Representation Families
    "convolutional neural networks",
    "recurrent neural networks",
    "long short-term memory",
    "gated recurrent",
    "graph neural networks",
    "graph convolutional",
    "graph attention",
    "knowledge graphs",
    "knowledge graph completion",
    "knowledge base question answering",
    "vision transformer",
    "diffusion models",
    "latent diffusion",
    "generative adversarial networks",
    "variational autoencoder",
    "normalizing flows",
    "mixture of experts",
    "state space models",
    "capsule networks",
    "siamese networks",
    "memory networks",
    "pointer networks",
    "conditional random fields",
    "support vector machines",
    "random forests",
    "decision trees",
    "hidden markov models",
    "bayesian networks",
    "gaussian processes",
    "spiking neural networks",
    "hyperbolic embeddings",
    "reinforcement learning",
    "inverse reinforcement learning",
    "offline reinforcement learning",
    "multi-agent reinforcement learning",
    "human feedback",
    "reward modeling",
    # Training, Compression, Security & Optimization Paradigms
    "federated learning",
    "continual learning",
    "lifelong learning",
    "catastrophic forgetting",
    "class-incremental",
    "meta-learning",
    "curriculum learning",
    "active learning",
    "semi-supervised learning",
    "weakly supervised",
    "distant supervision",
    "self-training",
    "co-training",
    "contrastive learning",
    "knowledge distillation",
    "network pruning",
    "weight pruning",
    "structured pruning",
    "post-training quantization",
    "quantization-aware training",
    "low-rank adaptation",
    "prompt tuning",
    "prefix tuning",
    "instruction tuning",
    "chain-of-thought",
    "in-context learning",
    "data augmentation",
    "domain adaptation",
    "unsupervised domain adaptation",
    "source-free domain adaptation",
    "multi-task learning",
    "adversarial training",
    "adversarial attacks",
    "adversarial examples",
    "backdoor attacks",
    "data poisoning",
    "membership inference",
    "differential privacy",
    "machine unlearning",
    "watermarking",
    "neural architecture search",
    "hyperparameter optimization",
    "bayesian optimization",
    "evolutionary algorithms",
    "monte carlo tree search",
    "beam search",
    "speculative decoding",
    "minimum bayes risk",
    # Vision, Multimodal, Domain & Scientific Subfields
    "computer vision",
    "image classification",
    "object detection",
    "semantic segmentation",
    "instance segmentation",
    "image generation",
    "text-to-image",
    "image captioning",
    "video understanding",
    "video captioning",
    "action recognition",
    "point cloud",
    "3d object detection",
    "neural radiance fields",
    "music generation",
    "audio classification",
    "multimodal learning",
    "vision-language",
    "cross-lingual transfer",
    "low-resource languages",
    "code-switching",
    "morphological inflection",
    "transliteration",
    "biomedical text",
    "clinical notes",
    "electronic health records",
    "medical imaging",
    "radiology reports",
    "drug discovery",
    "molecular property",
    "protein structure",
    "protein sequences",
    "genomic sequences",
    "legal documents",
    "judgment prediction",
    "financial forecasting",
    "social media",
    "recommendation systems",
    "collaborative filtering",
    "sequential recommendation",
    "time series forecasting",
    "anomaly detection",
    "traffic forecasting",
    "autonomous driving",
    "robotic manipulation",
    "embodied navigation",
    "tabular data",
    "table-to-text",
    "text-to-sql",
    "crowdsourcing",
    "eye-tracking",
    "brain decoding",
    "gender bias",
    "demographic parity",
    "counterfactual fairness",
    "out-of-distribution detection",
    "uncertainty quantification",
    "conformal prediction",
    "causal discovery",
    "causal inference",
)

_WORD_RE = re.compile(r"[a-z0-9]+")


def normalize_words(text: str) -> list[str]:
    """Ubah teks menjadi daftar token kata alfanumerik huruf kecil (tanda hubung dipisah)."""
    cleaned = text.lower().replace("-", " ").replace("/", " ")
    return _WORD_RE.findall(cleaned)


def canonical_token(tok: str) -> str:
    """Normalisasi ringan bentuk jamak sederhana agar pencocokan konsep konsisten."""
    if len(tok) > 4 and tok.endswith("ies"):
        return tok[:-3] + "y"
    if len(tok) > 4 and tok.endswith("es") and not tok.endswith(("ses", "xes", "zes")):
        pass
    if len(tok) > 3 and tok.endswith("s") and not tok.endswith(("ss", "us", "is")):
        return tok[:-1]
    return tok


def concept_token_tuple(concept: str) -> tuple[str, ...]:
    """Konversi frasa konsep menjadi tuple token kanonis."""
    return tuple(canonical_token(w) for w in normalize_words(concept))


def distinguishing_tokens(concept: str) -> set[str]:
    """Ambil kata pembeda spesifik dari konsep (di luar kata kepala metodologis umum)."""
    raw_tokens = [canonical_token(w) for w in normalize_words(concept)]
    specific = {
        t for t in raw_tokens
        if t not in {canonical_token(g) for g in GENERIC_HEAD_TOKENS} and len(t) >= 3
    }
    if not specific:
        specific = {t for t in raw_tokens if len(t) >= 3}
    return specific


def build_naive_negation_query(q_pos: str, q_neg: str) -> str:
    """Bentuk kueri M2 ('<Q_pos>, excluding <Q_neg>') dengan tanda baca kalimat alami."""
    stripped = q_pos.strip()
    neg_clean = q_neg.strip()
    if stripped.endswith("?"):
        return f"{stripped[:-1].rstrip()}, excluding {neg_clean}?"
    if stripped.endswith("."):
        return f"{stripped[:-1].rstrip()}, excluding {neg_clean}."
    return f"{stripped}, excluding {neg_clean}"


def build_concept_inverted_index(
    corpus: dict[str, dict[str, Any]],
    concepts: tuple[str, ...] = CS_CONCEPT_LEXICON,
    min_docs: int = MIN_CONCEPT_DOCS,
    max_docs: int = MAX_CONCEPT_DOCS,
) -> tuple[dict[str, list[str]], dict[str, set[str]], dict[str, set[str]]]:
    """Bangun inverted index konsep -> daftar doc_id di seluruh korpus (Layer 1).

    Mengembalikan:
      - concept_to_docs: {concept: sorted list of doc_ids (title U abstract)}
      - doc_to_title_concepts: {doc_id: set of concepts muncul di title}
      - doc_to_all_concepts: {doc_id: set of concepts muncul di title U abstract}
    """
    by_length: dict[int, dict[tuple[str, ...], str]] = {}
    for c in concepts:
        tup = concept_token_tuple(c)
        if tup:
            by_length.setdefault(len(tup), {})[tup] = c

    lengths = sorted(by_length.keys())
    raw_concept_docs: dict[str, list[str]] = {c: [] for c in concepts}
    raw_doc_title: dict[str, set[str]] = {}
    raw_doc_all: dict[str, set[str]] = {}

    for doc_id, doc in corpus.items():
        title_toks = [canonical_token(w) for w in normalize_words(doc.get("title") or "")]
        abs_toks = [canonical_token(w) for w in normalize_words(doc.get("text") or "")]

        title_matches: set[str] = set()
        n_t = len(title_toks)
        for k in lengths:
            if n_t < k:
                continue
            lookup = by_length[k]
            for i in range(n_t - k + 1):
                gram = tuple(title_toks[i : i + k])
                matched = lookup.get(gram)
                if matched is not None:
                    title_matches.add(matched)

        all_matches = set(title_matches)
        n_a = len(abs_toks)
        for k in lengths:
            if n_a < k:
                continue
            lookup = by_length[k]
            for i in range(n_a - k + 1):
                gram = tuple(abs_toks[i : i + k])
                matched = lookup.get(gram)
                if matched is not None:
                    all_matches.add(matched)

        if title_matches:
            raw_doc_title[doc_id] = title_matches
        if all_matches:
            raw_doc_all[doc_id] = all_matches
            for c in all_matches:
                raw_concept_docs[c].append(doc_id)

    valid_concepts = {
        c for c, dlist in raw_concept_docs.items()
        if min_docs <= len(dlist) <= max_docs
    }

    concept_to_docs = {
        c: sorted(raw_concept_docs[c])
        for c in concepts
        if c in valid_concepts
    }
    doc_to_title_concepts = {
        did: (cset & valid_concepts)
        for did, cset in raw_doc_title.items()
        if (cset & valid_concepts)
    }
    doc_to_all_concepts = {
        did: (cset & valid_concepts)
        for did, cset in raw_doc_all.items()
        if (cset & valid_concepts)
    }
    return concept_to_docs, doc_to_title_concepts, doc_to_all_concepts


def l2_normalize_rows(mat: np.ndarray) -> np.ndarray:
    """Normalisasi L2 per baris dengan pelindung pembagian nol 1e-12."""
    norms = np.maximum(np.linalg.norm(mat, axis=1, keepdims=True), 1e-12)
    return (mat / norms).astype(np.float32, copy=False)


def _has_token_conflict(spec_tokens: set[str], target_tokens: set[str]) -> bool:
    """Periksa apakah ada irisan kata atau prefiks akar kata (>= 5 huruf) dengan teks positif/judul gold."""
    if spec_tokens & target_tokens:
        return True
    for s in spec_tokens:
        if len(s) >= 5:
            stem = s[:5]
            for t in target_tokens:
                if len(t) >= 5 and t.startswith(stem):
                    return True
    return False


def construct_litsearch_neg(
    corpus: dict[str, dict[str, Any]],
    queries: dict[str, dict[str, Any]],
    qrels: dict[str, dict[str, int]],
    doc_ids: list[str],
    query_ids: list[str],
    doc_emb: np.ndarray,
    q_pos_emb: np.ndarray,
    top_k_distractors: int = TOP_K_DISTRACTORS,
    seed: int = SPLIT_SEED,
    val_ratio: float = VAL_RATIO,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], dict[str, Any]]:
    """Bangun himpunan uji LitSearch-Neg dengan protokol validasi 3 lapis + split deterministik Val/Test.

    Lapis 1 (Concept-Level Violators): violator_doc_ids mencakup seluruh dokumen korpus yang memuat konsep Q_neg.
    Lapis 2 (0% Gold & Q_pos Overlap): Q_neg bebas konflik dengan Q_pos dan 0% beririsan dengan gold papers.
    Lapis 3 (Domain Sanity & Stratified Split): Konsep taksonomi CS nyata + split seed=42 (20% Val / 80% Test).
    """
    t0 = time.perf_counter()
    concept_to_docs, doc_to_title_concepts, doc_to_all_concepts = build_concept_inverted_index(corpus)
    concept_doc_sets = {c: set(dlist) for c, dlist in concept_to_docs.items()}
    concept_spec_tokens = {c: distinguishing_tokens(c) for c in concept_to_docs}

    d_hat = l2_normalize_rows(doc_emb)
    q_hat = l2_normalize_rows(q_pos_emb)
    scores_all = q_hat @ d_hat.T  # (597, 64183)

    doc_id_arr = np.array(doc_ids, dtype=object)
    all_records: list[dict[str, Any]] = []
    skipped_no_distractor = 0
    skipped_no_clean_concept = 0

    for q_idx, qid in enumerate(query_ids):
        q_obj = queries[qid]
        q_pos_text = q_obj["text"]
        gold_set = set(qrels[qid].keys())

        row_scores = scores_all[q_idx]
        top10_indices = np.argpartition(-row_scores, kth=10)[:10]
        top10_indices = top10_indices[np.argsort(-row_scores[top10_indices])]
        top10_doc_ids = [str(doc_id_arr[i]) for i in top10_indices]

        top_k_non_gold = [
            (rank_1based, did)
            for rank_1based, did in enumerate(top10_doc_ids[:top_k_distractors], start=1)
            if did not in gold_set
        ]
        top10_non_gold_ids = [did for did in top10_doc_ids if did not in gold_set]

        if not top_k_non_gold:
            skipped_no_distractor += 1
            continue

        q_pos_tokens = {canonical_token(w) for w in normalize_words(q_pos_text)}
        gold_title_tokens: set[str] = set()
        gold_all_tokens: set[str] = set()
        for gid in gold_set:
            gdoc = corpus[gid]
            gt = {canonical_token(w) for w in normalize_words(gdoc.get("title") or "")}
            ga = {canonical_token(w) for w in normalize_words(gdoc.get("text") or "")}
            gold_title_tokens |= gt
            gold_all_tokens |= gt | ga

        # Kumpulkan kandidat konsep dari hard distractors di Top-5
        candidate_concepts: dict[str, tuple[int, bool, str]] = {}
        for rank_1based, did in top_k_non_gold:
            in_title_set = doc_to_title_concepts.get(did, set())
            in_all_set = doc_to_all_concepts.get(did, set())
            for c in in_all_set:
                in_title = c in in_title_set
                prev = candidate_concepts.get(c)
                if prev is None:
                    candidate_concepts[c] = (rank_1based, in_title, did)
                else:
                    prev_rank, prev_title, prev_did = prev
                    if (in_title and not prev_title) or (in_title == prev_title and rank_1based < prev_rank):
                        candidate_concepts[c] = (rank_1based, in_title, did)

        best_choice: tuple[tuple[int, int, int, int, str], str, int, bool, str] | None = None
        for c, (src_rank, in_title, src_did) in candidate_concepts.items():
            v_set = concept_doc_sets[c]
            spec_toks = concept_spec_tokens[c]

            # Lapis 2a: 0% konflik dengan teks Q_pos
            if _has_token_conflict(spec_toks, q_pos_tokens):
                continue

            # Lapis 2b: 0% irisan dengan gold papers (himpunan violator, judul gold, maupun seluruh kata pembeda di abstrak gold)
            if gold_set & v_set:
                continue
            if _has_token_conflict(spec_toks, gold_title_tokens):
                continue
            if spec_toks.issubset(gold_all_tokens):
                continue

            # Hitung berapa hard distractor di Top-10 yang memuat konsep negatif ini
            top10_violator_hits = sum(1 for did in top10_non_gold_ids if did in v_set)
            if top10_violator_hits == 0:
                continue

            # Prioritas deterministik:
            # 1) Muncul di judul hard distractor Top-5 (1 vs 0)
            # 2) Jumlah distractor Top-10 yang terkena konsep tersebut
            # 3) Peringkat kemunculan lebih tinggi (-src_rank)
            # 4) Frasa konsep multi-kata (len(concept_token_tuple(c)))
            # 5) Tie-breaker alfabetis terbalik agar deterministik penuh
            priority = (
                1 if in_title else 0,
                top10_violator_hits,
                -src_rank,
                min(len(concept_token_tuple(c)), 3),
                c,
            )
            if best_choice is None or priority > best_choice[0]:
                best_choice = (priority, c, src_rank, in_title, src_did)

        if best_choice is None:
            skipped_no_clean_concept += 1
            continue

        _, chosen_concept, src_rank, in_title, src_did = best_choice
        violator_ids = concept_to_docs[chosen_concept]

        # Assertion mutlak Lapis 2 (0% Gold Overlap)
        assert len(gold_set & set(violator_ids)) == 0, (
            f"Konflik gold terdeteksi pada {qid} untuk konsep {chosen_concept}"
        )

        q_naive = build_naive_negation_query(q_pos_text, chosen_concept)
        record = {
            "query_id": qid,
            "q_pos": q_pos_text,
            "q_neg": chosen_concept,
            "q_naive_not": q_naive,
            "gold_ids": sorted(gold_set),
            "violator_doc_ids": violator_ids,
            "violator_count": len(violator_ids),
            "source_distractor_id": src_did,
            "source_distractor_rank": int(src_rank),
            "source_match_location": "title" if in_title else "abstract",
            "specificity": int(q_obj["specificity"]),
            "quality": int(q_obj["quality"]),
            "query_set": str(q_obj["query_set"]),
        }
        all_records.append(record)

    # Lapis 3: Stratified deterministic split (seed=42) berdasarkan specificity (0 vs 1)
    rng = np.random.default_rng(seed)
    val_records: list[dict[str, Any]] = []
    test_records: list[dict[str, Any]] = []

    for spec_val in (0, 1):
        group = [r for r in all_records if r["specificity"] == spec_val]
        perm = rng.permutation(len(group))
        n_val = int(round(len(group) * val_ratio))
        val_idx_set = set(perm[:n_val].tolist())
        for idx, rec in enumerate(group):
            if idx in val_idx_set:
                val_records.append({**rec, "split": "val"})
            else:
                test_records.append({**rec, "split": "test"})

    val_records.sort(key=lambda r: r["query_id"])
    test_records.sort(key=lambda r: r["query_id"])

    val_qids = {r["query_id"] for r in val_records}
    test_qids = {r["query_id"] for r in test_records}
    assert len(val_qids & test_qids) == 0, "Kebocoran ID kueri antara Validation dan Test!"

    elapsed = time.perf_counter() - t0
    audit = {
        "total_litsearch_queries": len(query_ids),
        "validated_litsearch_neg_queries": len(all_records),
        "val_count": len(val_records),
        "test_count": len(test_records),
        "val_ratio_actual": round(len(val_records) / max(len(all_records), 1), 4),
        "skipped_no_top5_distractor": skipped_no_distractor,
        "skipped_no_conflict_free_concept": skipped_no_clean_concept,
        "unique_negative_concepts_used": len({r["q_neg"] for r in all_records}),
        "title_sourced_pct": round(
            100.0 * sum(1 for r in all_records if r["source_match_location"] == "title")
            / max(len(all_records), 1),
            2,
        ),
        "mean_violator_docs_per_query": round(
            float(np.mean([r["violator_count"] for r in all_records])) if all_records else 0.0,
            2,
        ),
        "median_violator_docs_per_query": round(
            float(np.median([r["violator_count"] for r in all_records])) if all_records else 0.0,
            2,
        ),
        "gold_overlap_conflicts": 0,
        "val_test_id_overlap": 0,
        "seed": seed,
        "top_k_distractors": top_k_distractors,
        "active_lexicon_concepts_in_corpus": len(concept_to_docs),
        "construction_seconds": round(elapsed, 3),
    }
    return val_records, test_records, audit


def save_litsearch_neg_splits(
    val_records: list[dict[str, Any]],
    test_records: list[dict[str, Any]],
    audit: dict[str, Any],
    corpus: dict[str, dict[str, Any]],
    data_dir: Path = DATA_DIR,
    tables_dir: Path = TABLES_DIR,
) -> None:
    """Simpan file JSONL split Validation (20%), Test (80%), audit JSON, dan sampel sanity check CSV."""
    data_dir.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)

    val_path = data_dir / "negative_queries_val.jsonl"
    test_path = data_dir / "negative_queries_test.jsonl"
    audit_path = data_dir / "litsearch_neg_audit.json"
    sample_csv_path = tables_dir / "litsearch_neg_sanity_sample.csv"

    for path, rows in ((val_path, val_records), (test_path, test_records)):
        with path.open("w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    with audit_path.open("w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2, ensure_ascii=False)

    combined = sorted(val_records + test_records, key=lambda r: r["query_id"])
    step = max(len(combined) // 30, 1)
    sample_rows = combined[::step][:30]
    with sample_csv_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "query_id",
                "split",
                "specificity",
                "q_pos",
                "q_neg",
                "q_naive_not",
                "gold_first_title",
                "distractor_rank",
                "distractor_location",
                "distractor_title",
                "violator_count",
            ],
        )
        writer.writeheader()
        for r in sample_rows:
            first_gid = r["gold_ids"][0]
            src_did = r["source_distractor_id"]
            writer.writerow({
                "query_id": r["query_id"],
                "split": r["split"],
                "specificity": r["specificity"],
                "q_pos": r["q_pos"],
                "q_neg": r["q_neg"],
                "q_naive_not": r["q_naive_not"],
                "gold_first_title": (corpus[first_gid].get("title") or "")[:120],
                "distractor_rank": r["source_distractor_rank"],
                "distractor_location": r["source_match_location"],
                "distractor_title": (corpus[src_did].get("title") or "")[:120],
                "violator_count": r["violator_count"],
            })

val_records, test_records, neg_audit = construct_litsearch_neg(
    corpus=corpus,
    queries=queries,
    qrels=qrels,
    doc_ids=doc_ids,
    query_ids=query_ids,
    doc_emb=doc_emb,
    q_pos_emb=q_pos_emb,
)
save_litsearch_neg_splits(
    val_records=val_records,
    test_records=test_records,
    audit=neg_audit,
    corpus=corpus,
    data_dir=DATA_DIR,
    tables_dir=TABLES_DIR,
)

print("=== HASIL KONSTRUKSI & AUDIT 3 LAPIS LITSEARCH-NEG ===")
print(json.dumps(neg_audit, indent=2))
print("\nContoh 3 pasangan (Q_pos, Q_neg, M2) tervalidasi:")
for sample in val_records[:3]:
    print(f"  - [{sample['query_id']} | {sample['split']}] Q_neg = {sample['q_neg']!r} (violators={sample['violator_count']})")
    print(f"    M2: {sample['q_naive_not']}")

=== HASIL KONSTRUKSI & AUDIT 3 LAPIS LITSEARCH-NEG ===
{
  "total_litsearch_queries": 597,
  "validated_litsearch_neg_queries": 480,
  "val_count": 96,
  "test_count": 384,
  "val_ratio_actual": 0.2,
  "skipped_no_top5_distractor": 0,
  "skipped_no_conflict_free_concept": 117,
  "unique_negative_concepts_used": 108,
  "title_sourced_pct": 44.58,
  "mean_violator_docs_per_query": 1409.95,
  "median_violator_docs_per_query": 550.0,
  "gold_overlap_conflicts": 0,
  "val_test_id_overlap": 0,
  "seed": 42,
  "top_k_distractors": 5,
  "active_lexicon_concepts_in_corpus": 184,
  "construction_seconds": 15.15
}

Contoh 3 pasangan (Q_pos, Q_neg, M2) tervalidasi:
  - [q_000 | val] Q_neg = 'machine translation' (violators=5765)
    M2: Are there any research papers on methods to compress large-scale language models using task-agnostic knowledge distillation techniques, excluding machine translation?
  - [q_003 | val] Q_neg = 'social media' (violators=1412)
    M2: Are there any tools or studies t

In [6]:
# === SEL 6: Enkode Q_neg & Q_naive_not (adhoc_query), Verifikasi Akhir & Buat ZIP Unduhan ===
import zipfile

# Wajib aktifkan kembali adapter adhoc_query untuk seluruh kueri negatif & kueri M2!
model.set_active_adapters("adhoc_query")
active_neg_adapter = str(model.active_adapters)
print("Adapter aktif untuk Q_neg & M2:", active_neg_adapter)
assert "adhoc_query" in active_neg_adapter and "proximity" not in active_neg_adapter, (
    f"GAGAL ASSERT ADAPTER KUERI NEGATIF! Terdeteksi: {active_neg_adapter}"
)

all_neg_ordered = sorted(val_records + test_records, key=lambda r: r["query_id"])
neg_query_ids = [r["query_id"] for r in all_neg_ordered]
q_neg_texts = [r["q_neg"] for r in all_neg_ordered]
q_naive_texts = [r["q_naive_not"] for r in all_neg_ordered]

torch.cuda.synchronize()
t0_qneg = time.perf_counter()
q_neg_emb = encode_texts_fp16(q_neg_texts, batch_size=64, desc="Encoding Q_neg (adhoc_query)")
q_naive_not_emb = encode_texts_fp16(q_naive_texts, batch_size=64, desc="Encoding Q_naive_not M2 (adhoc_query)")
torch.cuda.synchronize()
q_neg_seconds = round(time.perf_counter() - t0_qneg, 3)

np.save(EMBEDDINGS_DIR / "q_neg_emb.npy", q_neg_emb)
np.save(EMBEDDINGS_DIR / "q_naive_not_emb.npy", q_naive_not_emb)
with (EMBEDDINGS_DIR / "neg_query_ids.json").open("w", encoding="utf-8") as f:
    json.dump(neg_query_ids, f, indent=2)

encoding_log = {
    **env_info,
    "query_adapter": active_query_adapter,
    "doc_adapter": active_doc_adapter,
    "neg_query_adapter": active_neg_adapter,
    "batch_size_docs": 128,
    "batch_size_queries": 64,
    "fp16_autocast": True,
    "length_sorted_dynamic_padding": True,
    "max_seq_length": 512,
    "q_pos_encoding_seconds": q_pos_seconds,
    "doc_encoding_seconds": doc_encoding_seconds,
    "q_neg_and_naive_encoding_seconds": q_neg_seconds,
    "shapes": {
        "doc_emb": list(doc_emb.shape),
        "q_pos_emb": list(q_pos_emb.shape),
        "q_neg_emb": list(q_neg_emb.shape),
        "q_naive_not_emb": list(q_naive_not_emb.shape),
    },
    "litsearch_neg_audit": neg_audit,
}
with (EMBEDDINGS_DIR / "encoding_log.json").open("w", encoding="utf-8") as f:
    json.dump(encoding_log, f, indent=2)

# Verifikasi menyeluruh seluruh matriks & bebas konflik sebelum dikemas
for name, arr in (
    ("doc_emb", doc_emb),
    ("q_pos_emb", q_pos_emb),
    ("q_neg_emb", q_neg_emb),
    ("q_naive_not_emb", q_naive_not_emb),
):
    assert arr.dtype == np.float32, f"{name} bukan float32!"
    assert np.isfinite(arr).all(), f"{name} mengandung NaN/Inf!"
    assert float(np.linalg.norm(arr, axis=1).min()) > 0.0, f"{name} memiliki vektor nol!"

# Kemas seluruh file luaran Hari 2 sesuai struktur folder lokal metopen-specter2-litsearch/
zip_path = OUTPUT_ROOT / "day2_colab_outputs.zip"
files_to_pack = [
    (EMBEDDINGS_DIR / "doc_emb.npy", "embeddings/doc_emb.npy"),
    (EMBEDDINGS_DIR / "q_pos_emb.npy", "embeddings/q_pos_emb.npy"),
    (EMBEDDINGS_DIR / "q_neg_emb.npy", "embeddings/q_neg_emb.npy"),
    (EMBEDDINGS_DIR / "q_naive_not_emb.npy", "embeddings/q_naive_not_emb.npy"),
    (EMBEDDINGS_DIR / "doc_ids.json", "embeddings/doc_ids.json"),
    (EMBEDDINGS_DIR / "query_ids.json", "embeddings/query_ids.json"),
    (EMBEDDINGS_DIR / "neg_query_ids.json", "embeddings/neg_query_ids.json"),
    (EMBEDDINGS_DIR / "encoding_log.json", "embeddings/encoding_log.json"),
    (DATA_DIR / "negative_queries_val.jsonl", "data/litsearch/negative_queries_val.jsonl"),
    (DATA_DIR / "negative_queries_test.jsonl", "data/litsearch/negative_queries_test.jsonl"),
    (DATA_DIR / "litsearch_neg_audit.json", "data/litsearch/litsearch_neg_audit.json"),
    (TABLES_DIR / "litsearch_neg_sanity_sample.csv", "results/tables/litsearch_neg_sanity_sample.csv"),
]

print(f"\nMengemas 12 file luaran Hari 2 ke {zip_path}...")
with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src_file, arc_name in files_to_pack:
        assert src_file.exists(), f"File hilang sebelum zip: {src_file}"
        zf.write(src_file, arcname=arc_name)

zip_mb = round(zip_path.stat().st_size / (1024 ** 2), 2)
print(f"[SELESAI 100%] Arsip siap unduh: {zip_path} ({zip_mb} MB)")
print("Silakan unduh day2_colab_outputs.zip dari Google Drive, ekstrak langsung di folder metopen-specter2-litsearch/, lalu beri tahu saya untuk validasi akhir Hari 2!")

Adapter aktif untuk Q_neg & M2: Stack[adhoc_query]


Encoding Q_neg (adhoc_query):   0%|          | 0/8 [00:00<?, ?it/s]

Encoding Q_naive_not M2 (adhoc_query):   0%|          | 0/8 [00:00<?, ?it/s]


Mengemas 12 file luaran Hari 2 ke /content/drive/MyDrive/metopen_specter2_litsearch_day2/day2_colab_outputs.zip...
[SELESAI 100%] Arsip siap unduh: /content/drive/MyDrive/metopen_specter2_litsearch_day2/day2_colab_outputs.zip (167.95 MB)
Silakan unduh day2_colab_outputs.zip dari Google Drive, ekstrak langsung di folder metopen-specter2-litsearch/, lalu beri tahu saya untuk validasi akhir Hari 2!
